# 01 · Data audit

**Purpose:** before any modeling, check that the data are what we think they are.
Run `python scripts/download_data.py` and `python scripts/build_dataset.py` first.

This notebook contains no analysis logic; it only reads files produced by the pipeline,
so what you see here is exactly what the report uses.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
TABLES, FIGS, PROC = ROOT / "reports/tables", ROOT / "reports/figures", ROOT / "data/processed"
def show_fig(name):
    p = FIGS / name
    display(Image(str(p)) if p.exists() else Markdown(f"_{name} not generated yet_"))
def table(name):
    p = TABLES / f"{name}.csv"
    return pd.read_csv(p) if p.exists() else Markdown(f"_{name} not generated yet_")

## Download metadata
Where each file came from, when, and its hash. The API key is redacted.

In [ ]:
manifest = json.loads((PROC / "manifest.json").read_text())
for name, info in manifest["inputs"].items():
    print(name, json.dumps(info["metadata"], indent=1)[:900], "\n")

## Validation checks
Row counts, date ranges, missing values, units, spacing, weekend rows.

In [ ]:
display(Markdown((TABLES / "data_validation.md").read_text()))

## Missing prices
Weekdays with no Henry Hub price. Most should be market holidays; check the weekday column.

In [ ]:
mp = table("missing_prices"); mp

## Release alignment: the most important check
For every row, `first_eligible_price_date` must be **after** `report_release_date`, and
`price_before_release_date` **before** it. Look especially at holiday and verified weeks.

In [ ]:
al = pd.read_csv(TABLES / "release_alignment_table.csv", parse_dates=["storage_week_ending_date","report_release_date","first_eligible_price_date","price_before_release_date"])
ok = al[al["alignment_ok"]]
print("violations:", int((ok["first_eligible_price_date"] <= ok["report_release_date"]).sum()))
print(al["release_date_source"].value_counts())
al[al["release_date_source"] != "rule_nominal"].tail(10)

In [ ]:
al.sample(8, random_state=0).sort_values("report_release_date")

## Coverage and figures

In [ ]:
table("audit_coverage")

In [ ]:
for f in ["fig01_storage_level.png", "fig02_weekly_change.png", "fig04_henry_hub_price.png"]: show_fig(f)